## boilerplate

In [1]:
import torch
import torchaudio
import numpy as np
import os
import gc
import re
from pathlib import Path
from natsort import natsorted
import librosa
from sklearn.metrics import pairwise_distances
import warnings
print(f"Torch Version: {torch.__version__}")
# This should now be 2.6.0 or higher
print(f"Is CUDA available: {torch.cuda.is_available()}")
warnings.filterwarnings("ignore")



BATCH_SIZE = 16  # Can increase since 2s clips are small
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device is {DEVICE}")
DTYPE = torch.float32  # to ensure comaptibility across all models or 
BASE_DIR = Path("/home/amin/Research/Representation/Movie/data/Setareh/Data")
OUTPUT_PATH = Path ("/home/amin/Research/Representation/Movie/outputs/model_embeddings")


def extract_part_number(filename):
    """
    Extract part number from filename like 'video_part_001.mp4' or 'audio_part_042.wav'
    Returns the number as int, or None if not found
    """
    match = re.search(r'part[_-]?(\d+)', filename.stem, re.IGNORECASE)
    if match:
        return int(match.group(1))
    return None

all_videos = []
all_audios = []

for run_idx in range(1, 19):  # Video1 through Video18
    vid_chunk_dir = BASE_DIR / f"Video{run_idx}" / f"Video{run_idx}_chunks"
    aud_chunk_dir = BASE_DIR / f"Audio{run_idx}" / f"Audio{run_idx}_chunks"
    
    if not vid_chunk_dir.exists() or not aud_chunk_dir.exists():
        print(f"Warning: Missing dir for video {run_idx}")
        continue
    
    vids = natsorted(vid_chunk_dir.glob("*.mp4"))
    auds = natsorted(aud_chunk_dir.glob("*.wav"))
    if len(vids) == 0 or len(auds) == 0:
        print(f"Warning: No chunks found for video {run_idx}")
        continue
    
    vid_parts = {extract_part_number(v): v for v in vids if extract_part_number(v) is not None}
    aud_parts = {extract_part_number(a): a for a in auds if extract_part_number(a) is not None}
    
    common_parts = sorted(set(vid_parts.keys()) & set(aud_parts.keys()))
    
    for part_num in common_parts:
        all_videos.append(vid_parts[part_num])
        all_audios.append(aud_parts[part_num])
    
    print(f"Video {run_idx}: {len(common_parts)} paired chunks matched.")

print(f"\nTotal segments matched: {len(all_videos)}")

def load_video_fixed(path, num_frames=16):
    from decord import VideoReader, cpu
    import decord
    decord.bridge.set_bridge("torch")
    vr = VideoReader(str(path), ctx=cpu(0))
    idx = np.linspace(0, len(vr) - 1, num_frames).astype(np.int64)
    frames = vr.get_batch(idx) 
    return frames.permute(0, 3, 1, 2) 

def load_audio_fixed(path, target_len, sr=16000):
    # Load with librosa (most compatible option)
    wav, _ = librosa.load(path, sr=sr, mono=True)
    wav = torch.from_numpy(wav)
    
    # Pad or trim to target length
    if wav.numel() < target_len:
        wav = torch.nn.functional.pad(wav, (0, target_len - wav.numel()))
    else:
        wav = wav[:target_len]
    
    return wav

Torch Version: 2.10.0+cu128
Is CUDA available: True
device is cuda
Video 1: 106 paired chunks matched.
Video 2: 110 paired chunks matched.
Video 3: 93 paired chunks matched.
Video 4: 31 paired chunks matched.
Video 5: 41 paired chunks matched.
Video 6: 113 paired chunks matched.
Video 7: 129 paired chunks matched.
Video 8: 124 paired chunks matched.
Video 9: 41 paired chunks matched.
Video 10: 90 paired chunks matched.
Video 11: 92 paired chunks matched.
Video 12: 101 paired chunks matched.
Video 13: 70 paired chunks matched.
Video 14: 41 paired chunks matched.
Video 15: 116 paired chunks matched.
Video 16: 115 paired chunks matched.
Video 17: 127 paired chunks matched.
Video 18: 41 paired chunks matched.

Total segments matched: 1581


## VideoMAE-Large Extraction

In [2]:
from transformers import AutoModel, AutoImageProcessor
import os

VIDEOMAE_PATH = "/home/amin/hf_models/manual/models--OpenGVLab--VideoMAEv2-Large/snapshots/c1423ed94bb01d80a3f5ce5bc39f6026a0f4828c"

try:
    model_v = AutoModel.from_pretrained(VIDEOMAE_PATH, local_files_only=True, trust_remote_code=True).to(DTYPE).to(DEVICE).eval()
    processor_v = AutoImageProcessor.from_pretrained(VIDEOMAE_PATH, local_files_only=True)
    print("VideoMAE Ready (Local)")
except Exception as e:
    print(f"Local load failed: {e}. Downloading from Hub...")
    model_v = AutoModel.from_pretrained("OpenGVLab/VideoMAEv2-Large", trust_remote_code=True).to(DTYPE).to(DEVICE).eval()
    processor_v = AutoImageProcessor.from_pretrained("OpenGVLab/VideoMAEv2-Large")
    print("VideoMAE Ready (Hub)")

Local load failed: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/home/amin/hf_models/manual/models--OpenGVLab--VideoMAEv2-Large/snapshots/c1423ed94bb01d80a3f5ce5bc39f6026a0f4828c'. Use `repo_type` argument if needed.. Downloading from Hub...


A new version of the following files was downloaded from https://huggingface.co/OpenGVLab/VideoMAEv2-Large:
- modeling_config.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/OpenGVLab/VideoMAEv2-Large:
- modeling_videomaev2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


VideoMAE Ready (Hub)


In [3]:
vmae_embs = []
with torch.inference_mode():
    for i, vp in enumerate(all_videos):
        frames = load_video_fixed(vp)
        inputs = processor_v(list(frames), return_tensors="pt")
        
        # Fix: Permute from [B, T, C, H, W] to [B, C, T, H, W]
        pixel_values = inputs['pixel_values']
        if pixel_values.ndim == 5 and pixel_values.shape[1] > pixel_values.shape[2]:
            # Detected [B, T, C, H, W], permute to [B, C, T, H, W]
            pixel_values = pixel_values.permute(0, 2, 1, 3, 4)
            inputs['pixel_values'] = pixel_values
        
        inputs = {k: v.to(DTYPE).to(DEVICE) for k, v in inputs.items()}
        out = model_v(**inputs)
        
        # Handle different return types
        if isinstance(out, torch.Tensor):
            # Direct tensor output - use as is or pool if needed
            if out.ndim == 3:  # [B, seq_len, hidden_dim]
                h = out.mean(dim=1)  # Pool over sequence
            else:
                h = out
        elif hasattr(out, 'last_hidden_state'):
            h = out.last_hidden_state.mean(dim=1)
        elif hasattr(out, 'pooler_output'):
            h = out.pooler_output
        else:
            # Fallback: assume first element is the hidden states
            h = out[0].mean(dim=1) if out[0].ndim == 3 else out[0]
        if (i + 1) % 50 == 0:
            print(f"Processed {i + 1}/{len(all_videos)} segments...")
        vmae_embs.append(h.cpu())

videomae_out = torch.cat(vmae_embs).numpy()
print(f"VideoMAE embeddings shape: {videomae_out.shape}")

RuntimeError: CUDA error: no kernel image is available for execution on the device
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# Save VideoMAE embeddings
np.save(OUTPUT_PATH / "videomae-large_embeddings.npy", videomae_out)
print(f"Saved VideoMAE embeddings to {OUTPUT_PATH / 'videomae-large_embeddings.npy'}")

In [ ]:
del model_v, processor_v, vmae_embs
gc.collect()
torch.cuda.empty_cache()
print("VideoMAE Cleared.")

## WavLM Extraction

In [ ]:
from transformers import WavLMModel, Wav2Vec2FeatureExtractor
import os
WAVLM_PATH = "/home/amin/hf_models/manual/models--microsoft--wavlm-large/snapshots/c1423ed94bb01d80a3f5ce5bc39f6026a0f4828c"

os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"

try:
    model_a = WavLMModel.from_pretrained(WAVLM_PATH, local_files_only=True, use_safetensors= True).to(DTYPE).to(DEVICE).eval()
    processor_a = Wav2Vec2FeatureExtractor.from_pretrained(WAVLM_PATH, local_files_only=True)
    print("WavLM Model & Processor Ready")
except Exception as e:
    print(f"Error loading WavLM locally: {e}")
    print("Downloading from HuggingFace...")
    model_a = WavLMModel.from_pretrained("microsoft/wavlm-large").to(DTYPE).to(DEVICE).eval()
    processor_a = Wav2Vec2FeatureExtractor.from_pretrained("microsoft/wavlm-large")
    print("WavLM Model & Processor Ready")

In [ ]:
wavlm_embs = []
T = 2.0  # 2s chunks
target_len = int(T * 16000)

with torch.inference_mode():
    for i , ap in enumerate(all_audios):
        wav = load_audio_fixed(ap, target_len)
        inp = processor_a(wav.numpy(), sampling_rate=16000, return_tensors="pt").to(DTYPE).to(DEVICE)
        out = model_a(**inp)
        # Mean pool over temporal dimension
        h = out.last_hidden_state.mean(dim=1)
        wavlm_embs.append(h.cpu())
        if (i + 1) % 50 == 0:
            print(f"Processed {i + 1}/{len(all_audios)} audio segments...")

wavlm_out = torch.cat(wavlm_embs).numpy()
print(f"WavLM embeddings shape: {wavlm_out.shape}")

In [ ]:
# Save WavLM embeddings
np.save(OUTPUT_PATH / "wavlm-large_embeddings.npy", wavlm_out)
print(f"Saved WavLM embeddings to {OUTPUT_PATH / 'wavlm-large_embeddings.npy'}")

In [ ]:
del model_a, processor_a, wavlm_embs
gc.collect()
torch.cuda.empty_cache()
print("WavLM Cleared.")

## AudioMAE-base Extraction

### **Methods Draft: AudioMAE Feature Extraction**

**Model & Architecture**

* **Model:** AudioMAE (Masked Autoencoders that Listen) [Huang et al., 2022].
* **Backbone:** ViT-Base (M params, ,  blocks).
* **Weights:** Pre-trained on AudioSet-2M [Gemmeke et al., 2017].
* **Checkpoint Choice:** **Self-Supervised** (not fine-tuned). We chose this to preserve rich acoustic variance and structural details (reconstruction objective) rather than collapsing features for classification, ensuring better alignment with biological neural activity.

**Implementation & Preprocessing**

* **Wrapper:** Used `hance-ai` wrapper in Hugging Face `transformers` to ensure strict adherence to original spectrogram protocols.
* **Input:** 2-second audio clips from video stimuli.
* **Spectrogram Specs:**
* Resample to  kHz.
* Mel-spectrogram: Window ms, Hop ms,  Mel bands.
* Patches:  grid (Time  Freq).



**Feature Extraction (Pooling)**

* **Goal:** Align granular model output with the averaged BOLD response (2s trial duration).
* **Extraction Point:** Final Transformer Block.
* **Tensor Shape:**  (Channels, Freq patches, Time patches).
* **Strategy:** **Global Average Pooling** across both  and  dimensions.
* **Final Output:** A single **768-dimensional vector** per stimulus, used to construct the Model RDM.

**Key References**

> * Huang, P. Y., et al. (2022). "Masked Autoencoders that Listen." *NeurIPS*.
> * Gemmeke, J. F., et al. (2017). "Audio Set." *ICASSP*.
> 
>

In [3]:
from transformers import AutoModel
import torch
import torchaudio

# Path to the corrected download
AUDIOMAE_PATH = "/home/amin/hf_models/manual/AudioMAE"

# Load Model
model_mae = AutoModel.from_pretrained(
    AUDIOMAE_PATH, 
    local_files_only=True,
    trust_remote_code=True
).to(DTYPE).to(DEVICE).eval()

print(f"AudioMAE (Base) Ready")

AudioMAE (Base) Ready


In [9]:
# Get the ENCODER output, not decoder output
# The encoder should give you a global representation

audiomae_embs = []

print("Starting AudioMAE Extraction...")
with torch.inference_mode():
    for i, ap in enumerate(all_audios):
        # Check if model has an encoder attribute
        if hasattr(model_mae, 'encoder'):
            # Get encoder output directly
            outputs = model_mae.encoder.encode(ap, DEVICE)
        else:
            outputs = model_mae(ap)
        
        if i == 0:
            print(f"Output shape: {outputs.shape}")
        
        # Pool over ALL dimensions to get single vector
        # [768, 8, 64] -> [768] by averaging over patches and frequency
        while outputs.ndim > 1:
            outputs = outputs.mean(dim=1)  # Pool over freq and time
        
        embedding = outputs.unsqueeze(0)  # [768] -> [1, 768]
        
        if i == 0:
            print(f"Final embedding shape: {embedding.shape}")
            
        audiomae_embs.append(embedding.cpu())
        
        if (i + 1) % 50 == 0:
            print(f"Processed {i + 1}/{len(all_audios)} audio segments...")

audiomae_out = torch.cat(audiomae_embs, dim=0).numpy()
print(f"AudioMAE embeddings shape: {audiomae_out.shape}")  # Should be (1581, 768)

Starting AudioMAE Extraction...
Output shape: torch.Size([768, 8, 64])
Final embedding shape: torch.Size([1, 768])
Processed 50/1581 audio segments...
Processed 100/1581 audio segments...
Processed 150/1581 audio segments...
Processed 200/1581 audio segments...
Processed 250/1581 audio segments...
Processed 300/1581 audio segments...
Processed 350/1581 audio segments...
Processed 400/1581 audio segments...
Processed 450/1581 audio segments...
Processed 500/1581 audio segments...
Processed 550/1581 audio segments...
Processed 600/1581 audio segments...
Processed 650/1581 audio segments...
Processed 700/1581 audio segments...
Processed 750/1581 audio segments...
Processed 800/1581 audio segments...
Processed 850/1581 audio segments...
Processed 900/1581 audio segments...
Processed 950/1581 audio segments...
Processed 1000/1581 audio segments...
Processed 1050/1581 audio segments...
Processed 1100/1581 audio segments...
Processed 1150/1581 audio segments...
Processed 1200/1581 audio segme

In [13]:
# Save AuidoMAE embeddings
np.save(OUTPUT_PATH / "AudioMAE_hf_embeddings.npy", audiomae_out)
print(f"Saved AuidoMAE embeddings to {OUTPUT_PATH / 'AudioMAE_hf_embeddings.npy'}")

Saved AuidoMAE embeddings to /home/amin/Research/Representation/Movie/outputs/model_embeddings/AudioMAE_hf_embeddings.npy


In [ ]:
del model_mae, audiomae_out
gc.collect()
torch.cuda.empty_cache()
print("AuidoMAE Cleared.")

AuidoMAE Cleared.


## PE-Core-Large Extraction

In [7]:
import torch
import torch.nn as nn
from torchvision.transforms.functional import to_pil_image
import core.vision_encoder.pe as pe
import core.vision_encoder.transforms as transforms
import os
import numpy as np

LOCAL_WEIGHTS_PATH = os.path.expanduser("/home/amin/.cache/huggingface/hub/models-facebook--PE-Core-L14/PE-Core-L14-336.pt")

try:
    model = pe.CLIP.from_config("PE-Core-L14-336", pretrained=False)
    state_dict = torch.load(LOCAL_WEIGHTS_PATH, map_location='cpu')
    if "model" in state_dict:
        state_dict = state_dict["model"]
    model.load_state_dict(state_dict, strict=True)
    print(f"Weights loaded successfully.")

    model = model.to(DEVICE, dtype=DTYPE).eval()

except Exception as e:
    print(f"\nFATAL ERROR: {e}")
    model = pe.CLIP.from_config("PE-Core-L14-336", pretrained=True)

resolution = int(model.image_size)
preprocess = transforms.get_image_transform(resolution)
print(f"Using transform with resolution: {resolution}")

Weights loaded successfully.
Using transform with resolution: 336


In [ ]:
from torchvision.transforms.functional import to_pil_image
import torch

# Ensure the output directory exists
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

pecore_embs = []
model.eval()

print(f"Starting extraction for {len(all_videos)} segments...")

with torch.inference_mode():
    for i, vp in enumerate(all_videos):
        # 1. Load 16 frames: returns [16, 3, H, W]
        frames = load_video_fixed(vp) 
        
        # 2. Preprocess: Convert tensor frames to PIL, apply transforms, stack, and move to GPU
        # PE-Core requires 336x336 normalization
        processed_frames = torch.stack([
            preprocess(to_pil_image(f)) for f in frames
        ]).to(DEVICE, dtype=DTYPE)
        
        # 3. Vision Tower Forward Pass
        # Returns global features for each frame [16, 1024]
        visual_features = model.visual(processed_frames) 
        
        # 4. Temporal Pooling
        # Mean pool the 16 frames to get one 1024-dim vector for the 2s segment
        h = visual_features.mean(dim=0, keepdim=True) # [1, 1024]
        
        pecore_embs.append(h.cpu())
        
        if (i + 1) % 50 == 0:
            print(f"Processed {i + 1}/{len(all_videos)} segments...")

# 5. Concatenate and Save
pecore_out = torch.cat(pecore_embs).numpy()
print(f"PE-Core-Large embeddings shape: {pecore_out.shape}")

Starting extraction for 1581 segments...
Processed 50/1581 segments...
Processed 100/1581 segments...
Processed 150/1581 segments...
Processed 200/1581 segments...
Processed 250/1581 segments...
Processed 300/1581 segments...
Processed 350/1581 segments...
Processed 400/1581 segments...
Processed 450/1581 segments...
Processed 500/1581 segments...
Processed 550/1581 segments...
Processed 600/1581 segments...
Processed 650/1581 segments...
Processed 700/1581 segments...
Processed 750/1581 segments...
Processed 800/1581 segments...
Processed 850/1581 segments...
Processed 900/1581 segments...
Processed 950/1581 segments...
Processed 1000/1581 segments...
Processed 1050/1581 segments...
Processed 1100/1581 segments...
Processed 1150/1581 segments...
Processed 1200/1581 segments...
Processed 1250/1581 segments...
Processed 1300/1581 segments...
Processed 1350/1581 segments...
Processed 1400/1581 segments...
Processed 1450/1581 segments...
Processed 1500/1581 segments...
Processed 1550/1581

In [ ]:
# Save PE-Core-Large embeddings
np.save(OUTPUT_PATH / "pecore-l14_embeddings.npy", pecore_out)
print(f"Saved to: {OUTPUT_PATH / 'pecore-l14_embeddings.npy'}")

Saved PE-Core-Large embeddings to /home/amin/Research/Representation/Movie/outputs/model_embeddings/pecore_large_embeddings.npy


In [10]:
del model, preprocess, pecore_out, h, visual_features, processed_frames, frames
gc.collect()
torch.cuda.empty_cache()
print("PE-Core-Large Cleared.")

PE-Core-Large Cleared.


## CLAP Extraction

In [20]:
from transformers import ClapModel, ClapProcessor
import os

CLAP_PATH = "/home/amin/hf_models/manual/models--laion--larger_clap_music_and_speech/snapshots/195c3a3e68faebb3e2088b9a79e79b43ddbda76b"

try:
    # Explicitly use the ClapModel class and disable safetensors check
    model_clap = ClapModel.from_pretrained(
        CLAP_PATH, 
        local_files_only=True, 
        use_safetensors=True
    ).to(DTYPE).to(DEVICE).eval()
    
    processor_clap = ClapProcessor.from_pretrained(
        CLAP_PATH, 
        local_files_only=True
    )
    print(" CLAP Model & Processor Ready (Loaded from .bin)")

except Exception as e:
    print(f" Failed to load CLAP: {e}")

 CLAP Model & Processor Ready (Loaded from .bin)


In [22]:
clap_embs = []
T = 2.0
target_len = int(T * 48000)  # CLAP uses 48kHz

with torch.inference_mode():
    for i, ap in enumerate(all_audios):
        wav = load_audio_fixed(ap, target_len, sr=48000)
        inputs = processor_clap(audios=wav.numpy(), sampling_rate=48000, return_tensors="pt")
        
        # Move tensors to device and dtype individually
        inputs = {k: v.to(DTYPE).to(DEVICE) if isinstance(v, torch.Tensor) else v 
                  for k, v in inputs.items()}
        
        out = model_clap.get_audio_features(**inputs)
        # CLAP audio features are already pooled, shape: (batch, hidden_size)
        clap_embs.append(out.cpu())
        
        if (i + 1) % 50 == 0:
            print(f"Processed {i + 1}/{len(all_audios)} audio segments...")

clap_out = torch.cat(clap_embs).numpy()
print(f"CLAP embeddings shape: {clap_out.shape}")

Processed 50/1581 audio segments...
Processed 100/1581 audio segments...
Processed 150/1581 audio segments...
Processed 200/1581 audio segments...
Processed 250/1581 audio segments...
Processed 300/1581 audio segments...
Processed 350/1581 audio segments...
Processed 400/1581 audio segments...
Processed 450/1581 audio segments...
Processed 500/1581 audio segments...
Processed 550/1581 audio segments...
Processed 600/1581 audio segments...
Processed 650/1581 audio segments...
Processed 700/1581 audio segments...
Processed 750/1581 audio segments...
Processed 800/1581 audio segments...
Processed 850/1581 audio segments...
Processed 900/1581 audio segments...
Processed 950/1581 audio segments...
Processed 1000/1581 audio segments...
Processed 1050/1581 audio segments...
Processed 1100/1581 audio segments...
Processed 1150/1581 audio segments...
Processed 1200/1581 audio segments...
Processed 1250/1581 audio segments...
Processed 1300/1581 audio segments...
Processed 1350/1581 audio segmen

In [ ]:
# Save CLAP embeddings
np.save(OUTPUT_PATH / "clap-large_embeddings.npy", clap_out)
print(f"Saved CLAP embeddings to {OUTPUT_PATH / 'clap-large_embeddings.npy'}")

Saved CLAP embeddings to /home/amin/Research/Representation/Movie/outputs/model_embeddings/clap_embeddings.npy


In [24]:
del model_clap, processor_clap, clap_embs
gc.collect()
torch.cuda.empty_cache()
print("CLAP Cleared.")

CLAP Cleared.


## Whisper Extraction

In [15]:
from transformers import WhisperModel, WhisperProcessor
import os

WHISPER_PATH = "/home/amin/hf_models/manual/models--openai--whisper-large-v3/snapshots/06f233fe06e710322aca913c1bc4249a0d71fce1"
LOCAL_FILES_ONLY = True

try:
    if LOCAL_FILES_ONLY:
        model_whisper = WhisperModel.from_pretrained(
            WHISPER_PATH, 
            local_files_only=True
        ).to(DTYPE).to(DEVICE).eval()
        
        processor_whisper = WhisperProcessor.from_pretrained(
            WHISPER_PATH, 
            local_files_only=True
        )
    print("Whisper Model & Processor Ready (Sharded Safetensors Loaded)")
except Exception as e:
    print(f"Error: {e}")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Whisper Model & Processor Ready (Sharded Safetensors Loaded)


In [16]:
whisper_embs = []
T = 2.0
target_len = int(T * 16000)

with torch.inference_mode():
    for i , ap in enumerate(all_audios):
        wav = load_audio_fixed(ap, target_len)
        inputs = processor_whisper(wav.numpy(), sampling_rate=16000, return_tensors="pt").to(DTYPE).to(DEVICE)
        # Get encoder outputs
        encoder_outputs = model_whisper.encoder(
            inputs.input_features,
            output_hidden_states=False,
            return_dict=True
        )
        # Mean pool over temporal dimension
        h = encoder_outputs.last_hidden_state.mean(dim=1)
        whisper_embs.append(h.cpu())
        if (i + 1) % 50 == 0:
            print(f"Processed {i + 1}/{len(all_audios)} audio segments...")

whisper_out = torch.cat(whisper_embs).numpy()
print(f"Whisper embeddings shape: {whisper_out.shape}")

Processed 50/1581 audio segments...
Processed 100/1581 audio segments...
Processed 150/1581 audio segments...
Processed 200/1581 audio segments...
Processed 250/1581 audio segments...
Processed 300/1581 audio segments...
Processed 350/1581 audio segments...
Processed 400/1581 audio segments...
Processed 450/1581 audio segments...
Processed 500/1581 audio segments...
Processed 550/1581 audio segments...
Processed 600/1581 audio segments...
Processed 650/1581 audio segments...
Processed 700/1581 audio segments...
Processed 750/1581 audio segments...
Processed 800/1581 audio segments...
Processed 850/1581 audio segments...
Processed 900/1581 audio segments...
Processed 950/1581 audio segments...
Processed 1000/1581 audio segments...
Processed 1050/1581 audio segments...
Processed 1100/1581 audio segments...
Processed 1150/1581 audio segments...
Processed 1200/1581 audio segments...
Processed 1250/1581 audio segments...
Processed 1300/1581 audio segments...
Processed 1350/1581 audio segmen

In [ ]:
# Save Whisper embeddings
np.save(OUTPUT_PATH / "whisper-large-v3_embeddings.npy", whisper_out)
print(f"Saved Whisper embeddings to {OUTPUT_PATH / 'whisper-large-v3_embeddings.npy'}")

Saved Whisper embeddings to /home/amin/Research/Representation/Movie/outputs/model_embeddings/whisper_embeddings.npy


In [18]:
del model_whisper, processor_whisper, whisper_embs
gc.collect()
torch.cuda.empty_cache()
print("Whisper Cleared.")

Whisper Cleared.


## Next steps:
VideoMAE-base videomae-small, and other sizes of PE-Core (small and base) and wav-lm ( small and base) and whisper (small and base) should be downloaded when available; along with different sizes of PE-Core (large and giant have already been downloaded) to make sure the RSA results are consistent. I also have to study the backbone and structure of the alignment and the cores of the models to make sure that the similarities are drawn between similar models in terms of structure and size


| **Size Tier** | **Cornerstone (PE-AV)** | **A-V Control (Speech)** | **A-V Control (General, no text)** | **Visual Control (Reconstructive)** | **Visual Control (Text-Aligned)** | **Audio Control (Reconstructive)** | **Audio Control (Text-Aligned)** | **Audio Speech Control (Structure)** |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **Small**<br>*(Audio: Base)* | **PE-AV-Small**<br>*(Audio: ViT-B, ~90M)*<br>*(Video: ViT-L, ~300M)* | **AV-HuBERT-Base**<br>*(ViT-Base, ~95M)* | **CAV-MAE-Sync**<br>*(ViT-Base, ~86M)*<br>*(2025 SOTA)* | **VideoMAE-v2-Large**<br>*(ViT-L, ~300M)*<br>*(Matches PE-L)* | **PE-Core-Large**<br>*(PE-Core-L14-336)*<br>*(~300M)* | **AudioMAE-Base**<br>*(ViT-B, ~86M)*<br>*(Perfect Match)* | **LAION-CLAP**<br>*(HTSAT-Base)*<br>*(~19M)* | **WavLM-Base+**<br>*(ViT-Base, ~95M)*<br>*(Structure Match)* |
| **Base**<br>*(Audio: Large)* | **PE-AV-Base**<br>*(Audio: ViT-L, ~210M)*<br>*(Video: ViT-L, ~300M)* | **AV-HuBERT-Large**<br>*(ViT-Large, ~316M)*<br>*(Good Match)* | *CAV-MAE-Sync*<br>*(Base: Under-match)* | **VideoMAE-v2-Large**<br>*(ViT-L, ~300M)*<br>*(Perfect Match)* | **PE-Core-Large**<br>*(PE-Core-L14-336)*<br>*(~300M)* | *AudioMAE-Base*<br>*(Mismatch: ~86M)* | **LAION-CLAP**<br>*(HTSAT-Large)*<br>*(or Large-Fuse)* | **WavLM-Large**<br>*(ViT-Large, ~317M)*<br>*(Structure Match)* |
| **Large**<br>*(Audio: Giant)* | **PE-AV-Large**<br>*(Audio: ViT-Giant, ~1.1B)*<br>*(Video: ViT-L, ~300M)* | *AV-HuBERT-Large*<br>*(Under-match)* | **ImageBind-Huge**<br>*(ViT-H, ~630M)*<br>*(Closest A-V Match)* | **VideoMAE-v2-Large**<br>*(ViT-L, ~300M)*<br>*(Perfect Match)* | **PE-Core-Large**<br>*(PE-Core-L14-336)*<br>*(~300M)* | *AudioMAE-Base*<br>*(Major Mismatch)* | *CLAP-Large*<br>*(Under-match)* | **Whisper-Large-v3**<br>*(ViT-Giant, ~1.55B)*<br>*(Size Match)* |
| **Giant**<br>*(Visual: Giant)* | **N/A**<br>*(Baseline Reference)* | *None* | **ImageBind-Huge**<br>*(ViT-H, ~630M)* | **VideoMAE-v2-Giant**<br>*(ViT-g, ~1B)* | **PE-Core-Giant**<br>*(PE-Core-G14)*<br>*(~1.9B)* | *None* | *None* | **Whisper-Large-v3**<br>*(~1.55B)* |